In [1]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os

# TensorFlow and Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, losses
from tensorflow.keras.datasets import mnist, cifar10
from tensorflow.keras.utils import plot_model

# Image processing
from PIL import Image
import imageio

# Additional utilities
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# GPU configuration
physical_devices = tf.config.experimental.list_physical_devices('GPU')
if len(physical_devices) > 0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print(f"GPU available: {physical_devices[0]}")
else:
    print("GPU not available, using CPU")

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Create output directories
os.makedirs('generated_images', exist_ok=True)
os.makedirs('models', exist_ok=True)
print("Output directories created!")


ModuleNotFoundError: No module named 'imageio'

In [ ]:
# Deep Convolutional GAN (DCGAN) Implementation

class DCGAN:
    """
    Deep Convolutional GAN implementation for image generation
    """
    
    def __init__(self, latent_dim=100, img_shape=(28, 28, 1)):
        self.latent_dim = latent_dim
        self.img_shape = img_shape
        self.img_rows, self.img_cols, self.channels = img_shape
        
        # Build and compile the discriminator
        self.discriminator = self.build_discriminator()
        self.discriminator.compile(
            loss='binary_crossentropy',
            optimizer=optimizers.Adam(0.0002, 0.5),
            metrics=['accuracy']
        )
        
        # Build the generator
        self.generator = self.build_generator()
        
        # For the combined model we will only train the generator
        self.discriminator.trainable = False
        
        # The discriminator takes generated images as input and determines validity
        z = layers.Input(shape=(self.latent_dim,))
        img = self.generator(z)
        validity = self.discriminator(img)
        
        # The combined model (stacked generator and discriminator)
        self.combined = models.Model(z, validity)
        self.combined.compile(
            loss='binary_crossentropy',
            optimizer=optimizers.Adam(0.0002, 0.5)
        )
        
        print("DCGAN initialized successfully!")
        print(f"Latent dimension: {latent_dim}")
        print(f"Image shape: {img_shape}")
    
    def build_generator(self):
        """
        Build the Generator Network
        
        Returns:
            Generator model
        """
        model = models.Sequential()
        
        # Foundation for 7x7 image
        model.add(layers.Dense(128 * 7 * 7, activation="relu", input_dim=self.latent_dim))
        model.add(layers.Reshape((7, 7, 128)))
        
        # Upsample to 14x14
        model.add(layers.UpSampling2D())
        model.add(layers.Conv2D(128, kernel_size=3, padding="same"))
        model.add(layers.BatchNormalization(momentum=0.8))
        model.add(layers.Activation("relu"))
        
        # Upsample to 28x28
        model.add(layers.UpSampling2D())
        model.add(layers.Conv2D(64, kernel_size=3, padding="same"))
        model.add(layers.BatchNormalization(momentum=0.8))
        model.add(layers.Activation("relu"))
        
        # Final convolutional layer
        model.add(layers.Conv2D(self.channels, kernel_size=3, padding="same"))
        model.add(layers.Activation("tanh"))
        
        print("Generator architecture:")
        model.summary()
        
        noise = layers.Input(shape=(self.latent_dim,))
        img = model(noise)
        
        return models.Model(noise, img)
    
    def build_discriminator(self):
        """
        Build the Discriminator Network
        
        Returns:
            Discriminator model
        """
        model = models.Sequential()
        
        model.add(layers.Conv2D(32, kernel_size=3, strides=2, 
                               input_shape=self.img_shape, padding="same"))
        model.add(layers.LeakyReLU(alpha=0.2))
        model.add(layers.Dropout(0.25))
        
        model.add(layers.Conv2D(64, kernel_size=3, strides=2, padding="same"))
        model.add(layers.ZeroPadding2D(padding=((0,1),(0,1))))
        model.add(layers.BatchNormalization(momentum=0.8))
        model.add(layers.LeakyReLU(alpha=0.2))
        model.add(layers.Dropout(0.25))
        
        model.add(layers.Conv2D(128, kernel_size=3, strides=2, padding="same"))
        model.add(layers.BatchNormalization(momentum=0.8))
        model.add(layers.LeakyReLU(alpha=0.2))
        model.add(layers.Dropout(0.25))
        
        model.add(layers.Conv2D(256, kernel_size=3, strides=1, padding="same"))
        model.add(layers.BatchNormalization(momentum=0.8))
        model.add(layers.LeakyReLU(alpha=0.2))
        model.add(layers.Dropout(0.25))
        
        model.add(layers.Flatten())
        model.add(layers.Dense(1, activation='sigmoid'))
        
        print("Discriminator architecture:")
        model.summary()
        
        img = layers.Input(shape=self.img_shape)
        validity = model(img)
        
        return models.Model(img, validity)
    
    def train(self, X_train, epochs, batch_size=128, save_interval=50):
        """
        Train the GAN
        
        Args:
            X_train: Training data
            epochs: Number of training epochs
            batch_size: Batch size for training
            save_interval: Interval for saving generated images
            
        Returns:
            Training history
        """
        # Rescale -1 to 1
        X_train = X_train / 127.5 - 1.
        
        # Adversarial ground truths
        valid = np.ones((batch_size, 1))
        fake = np.zeros((batch_size, 1))
        
        # Training history
        d_losses = []
        g_losses = []
        
        for epoch in range(epochs):
            
            # ---------------------
            #  Train Discriminator
            # ---------------------
            
            # Select a random batch of images
            idx = np.random.randint(0, X_train.shape[0], batch_size)
            imgs = X_train[idx]
            
            # Sample noise and generate a batch of new images
            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
            gen_imgs = self.generator.predict(noise, verbose=0)
            
            # Train the discriminator (real classified as ones and generated as zeros)
            d_loss_real = self.discriminator.train_on_batch(imgs, valid)
            d_loss_fake = self.discriminator.train_on_batch(gen_imgs, fake)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)
            
            # ---------------------
            #  Train Generator
            # ---------------------
            
            # Train the generator (wants discriminator to mistake images as real)
            g_loss = self.combined.train_on_batch(noise, valid)
            
            # Store losses
            d_losses.append(d_loss[0])
            g_losses.append(g_loss)
            
            # Print the progress
            if epoch % 100 == 0:
                print(f"Epoch {epoch}/{epochs} [D loss: {d_loss[0]:.4f}, acc.: {100*d_loss[1]:.2f}%] [G loss: {g_loss:.4f}]")
            
            # If at save interval => save generated image samples
            if epoch % save_interval == 0:
                self.save_imgs(epoch)
        
        return {'d_losses': d_losses, 'g_losses': g_losses}
    
    def save_imgs(self, epoch):
        """
        Save generated images
        
        Args:
            epoch: Current epoch number
        """
        r, c = 5, 5
        noise = np.random.normal(0, 1, (r * c, self.latent_dim))
        gen_imgs = self.generator.predict(noise, verbose=0)
        
        # Rescale images 0 - 1
        gen_imgs = 0.5 * gen_imgs + 0.5
        
        fig, axs = plt.subplots(r, c, figsize=(10, 10))
        cnt = 0
        for i in range(r):
            for j in range(c):
                if self.channels == 1:
                    axs[i,j].imshow(gen_imgs[cnt, :,:,0], cmap='gray')
                else:
                    axs[i,j].imshow(gen_imgs[cnt])
                axs[i,j].axis('off')
                cnt += 1
        
        fig.suptitle(f"Generated Images - Epoch {epoch}")
        plt.savefig(f"generated_images/epoch_{epoch}.png")
        plt.close()
    
    def generate_images(self, num_images=25):
        """
        Generate new images
        
        Args:
            num_images: Number of images to generate
            
        Returns:
            Generated images array
        """
        noise = np.random.normal(0, 1, (num_images, self.latent_dim))
        gen_imgs = self.generator.predict(noise, verbose=0)
        
        # Rescale images 0 - 1
        gen_imgs = 0.5 * gen_imgs + 0.5
        
        return gen_imgs

print("DCGAN class defined successfully!")
